# 305-01 · Subquery vs CTE (WITH): ¿cuál es más rápido?

> **300-Lab Experiment** | Edgar Cartolari Esteves | [github.com/ecartolariesteves/300-Lab](https://github.com/ecartolariesteves/300-Lab)

Una pregunta que escucho constantemente en equipos de datos: *¿uso subquery o CTE?*

En este notebook lo comprobamos con datos reales, midiendo tiempo de ejecución, comparando planes de query y evaluando legibilidad.

**TL;DR:** En el 90% de los casos el rendimiento es igual. Pero hay una excepción importante que cambia todo.

## 0. Setup y generación de datos sintéticos

In [ ]:
import pandas as pd
import numpy as np
import time
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display

pd.set_option('display.float_format', '{:.2f}'.format)
print('Libraries loaded ✓')

In [ ]:
# Generamos dataset sintético de ventas
np.random.seed(42)

N_ORDERS = 500_000
N_CUSTOMERS = 10_000
REGIONS = ['North', 'South', 'East', 'West']
SEGMENTS = ['Premium', 'Standard', 'Budget']

customers = pd.DataFrame({
    'customer_id': range(1, N_CUSTOMERS + 1),
    'region': np.random.choice(REGIONS, N_CUSTOMERS),
    'segment': np.random.choice(SEGMENTS, N_CUSTOMERS)
})

orders = pd.DataFrame({
    'order_id': range(1, N_ORDERS + 1),
    'customer_id': np.random.randint(1, N_CUSTOMERS + 1, N_ORDERS),
    'amount': np.round(np.random.exponential(scale=150, size=N_ORDERS), 2),
    'order_date': pd.date_range('2022-01-01', periods=N_ORDERS, freq='1min')[:N_ORDERS]
})

print(f'orders: {len(orders):,} filas')
print(f'customers: {len(customers):,} filas')
orders.head(3)

In [ ]:
# Cargamos en SQLite (simula un motor SQL — misma lógica aplica a BigQuery/PostgreSQL)
conn = sqlite3.connect(':memory:')
customers.to_sql('customers', conn, index=False, if_exists='replace')
orders.to_sql('orders', conn, index=False, if_exists='replace')

# Índice para acelerar JOINs (como tendrías en producción)
conn.execute('CREATE INDEX idx_orders_customer ON orders(customer_id)')
conn.execute('CREATE INDEX idx_customers_id ON customers(customer_id)')
conn.commit()
print('Database ready ✓')

## 1. Función de benchmark

In [ ]:
def benchmark(conn, query, runs=5, label=''):
    """Ejecuta una query N veces y devuelve estadísticas de tiempo (ms)"""
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        result = pd.read_sql_query(query, conn)
        elapsed = (time.perf_counter() - start) * 1000
        times.append(elapsed)
    stats = {
        'label': label,
        'min_ms': round(min(times), 1),
        'mean_ms': round(sum(times)/len(times), 1),
        'max_ms': round(max(times), 1),
        'rows': len(result)
    }
    return stats, result

results_summary = []
print('Benchmark function ready ✓')

## 2. Caso 1 — Agregación simple: Top clientes por gasto

In [ ]:
q_sub_1 = """
SELECT customer_id, total_spent
FROM (
    SELECT customer_id, SUM(amount) AS total_spent
    FROM orders
    GROUP BY customer_id
) ranked
ORDER BY total_spent DESC
LIMIT 10
"""

q_cte_1 = """
WITH customer_spending AS (
    SELECT customer_id, SUM(amount) AS total_spent
    FROM orders
    GROUP BY customer_id
)
SELECT customer_id, total_spent
FROM customer_spending
ORDER BY total_spent DESC
LIMIT 10
"""

s1, r1 = benchmark(conn, q_sub_1, label='Subquery - Caso 1')
c1, _  = benchmark(conn, q_cte_1, label='CTE - Caso 1')

results_summary.extend([s1, c1])
print(f"Subquery: {s1['mean_ms']} ms  |  CTE: {c1['mean_ms']} ms")
display(r1)

## 3. Caso 2 — Filtrado: clientes que superan la media

In [ ]:
q_sub_2 = """
SELECT customer_id, SUM(amount) AS total_spent
FROM orders
GROUP BY customer_id
HAVING SUM(amount) > (
    SELECT AVG(total) FROM (
        SELECT customer_id, SUM(amount) AS total
        FROM orders
        GROUP BY customer_id
    )
)
ORDER BY total_spent DESC
LIMIT 20
"""

q_cte_2 = """
WITH customer_totals AS (
    SELECT customer_id, SUM(amount) AS total_spent
    FROM orders
    GROUP BY customer_id
),
avg_spending AS (
    SELECT AVG(total_spent) AS avg_total FROM customer_totals
)
SELECT ct.customer_id, ct.total_spent
FROM customer_totals ct, avg_spending
WHERE ct.total_spent > avg_total
ORDER BY total_spent DESC
LIMIT 20
"""

s2, _ = benchmark(conn, q_sub_2, label='Subquery - Caso 2')
c2, _ = benchmark(conn, q_cte_2, label='CTE - Caso 2')

results_summary.extend([s2, c2])
print(f"Subquery: {s2['mean_ms']} ms  |  CTE: {c2['mean_ms']} ms")

## 4. Caso 3 — Multi-nivel: Ranking por región y segmento

In [ ]:
q_sub_3 = """
SELECT region, segment, customer_id, total_spent, rnk
FROM (
    SELECT region, segment, customer_id, total_spent,
           RANK() OVER (PARTITION BY region, segment ORDER BY total_spent DESC) AS rnk
    FROM (
        SELECT c.region, c.segment, o.customer_id, SUM(o.amount) AS total_spent
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        GROUP BY c.region, c.segment, o.customer_id
    ) aggregated
) ranked
WHERE rnk <= 3
"""

q_cte_3 = """
WITH order_totals AS (
    SELECT c.region, c.segment, o.customer_id, SUM(o.amount) AS total_spent
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.region, c.segment, o.customer_id
),
ranked AS (
    SELECT *,
           RANK() OVER (PARTITION BY region, segment ORDER BY total_spent DESC) AS rnk
    FROM order_totals
)
SELECT region, segment, customer_id, total_spent, rnk
FROM ranked
WHERE rnk <= 3
"""

s3, r3 = benchmark(conn, q_sub_3, label='Subquery - Caso 3')
c3, _  = benchmark(conn, q_cte_3, label='CTE - Caso 3')

results_summary.extend([s3, c3])
print(f"Subquery: {s3['mean_ms']} ms  |  CTE: {c3['mean_ms']} ms")
display(r3.head(12))

## 5. Caso 4 — Reutilización: misma lógica usada dos veces

> ⚠️ Este es el caso interesante. Cuando reutilizas la misma subexpresión, el CTE evita recalcularla.

In [ ]:
q_sub_4 = """
SELECT a.customer_id, a.total_spent,
       ROUND(a.total_spent * 100.0 / b.grand_total, 2) AS pct_of_total
FROM (
    SELECT customer_id, SUM(amount) AS total_spent
    FROM orders GROUP BY customer_id
) a
CROSS JOIN (
    SELECT SUM(total_spent) AS grand_total FROM (
        SELECT customer_id, SUM(amount) AS total_spent
        FROM orders GROUP BY customer_id
    )
) b
ORDER BY total_spent DESC
LIMIT 10
"""

q_cte_4 = """
WITH customer_spending AS (
    SELECT customer_id, SUM(amount) AS total_spent
    FROM orders GROUP BY customer_id
),
totals AS (
    SELECT SUM(total_spent) AS grand_total FROM customer_spending
)
SELECT cs.customer_id, cs.total_spent,
       ROUND(cs.total_spent * 100.0 / t.grand_total, 2) AS pct_of_total
FROM customer_spending cs, totals t
ORDER BY total_spent DESC
LIMIT 10
"""

s4, r4 = benchmark(conn, q_sub_4, label='Subquery - Caso 4')
c4, _  = benchmark(conn, q_cte_4, label='CTE - Caso 4')

results_summary.extend([s4, c4])
diff = round((s4['mean_ms'] - c4['mean_ms']) / s4['mean_ms'] * 100, 1)
print(f"Subquery: {s4['mean_ms']} ms  |  CTE: {c4['mean_ms']} ms")
print(f"→ CTE es {diff}% más rápido en este caso")
display(r4)

## 6. Visualización de resultados

In [ ]:
df_results = pd.DataFrame(results_summary)
df_results['tipo'] = df_results['label'].apply(lambda x: 'Subquery' if 'Subquery' in x else 'CTE')
df_results['caso'] = df_results['label'].apply(lambda x: x.split('Caso ')[-1])

casos = ['1', '2', '3', '4']
sub_times = [df_results[(df_results['tipo']=='Subquery') & (df_results['caso']==c)]['mean_ms'].values[0] for c in casos]
cte_times = [df_results[(df_results['tipo']=='CTE') & (df_results['caso']==c)]['mean_ms'].values[0] for c in casos]

x = np.arange(len(casos))
width = 0.35
labels = [
    'Caso 1\nAgregación simple',
    'Caso 2\nFiltro por media',
    'Caso 3\nMulti-nivel + rank',
    'Caso 4\nReutilización lógica'
]

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, sub_times, width, label='Subquery', color='#378ADD', alpha=0.85)
bars2 = ax.bar(x + width/2, cte_times, width, label='CTE (WITH)', color='#1D9E75', alpha=0.85)

ax.set_xlabel('Caso de prueba', fontsize=11)
ax.set_ylabel('Tiempo medio (ms)', fontsize=11)
ax.set_title('Subquery vs CTE — Tiempo de ejecución por caso', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{bar.get_height():.0f}ms', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{bar.get_height():.0f}ms', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('results/benchmark_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado en results/benchmark_chart.png')

## 7. Conclusiones

| | Rendimiento | Legibilidad | Reutilización |
|---|---|---|---|
| **Subquery** | ✅ Igual (casos 1-3) | ❌ Difícil de mantener | ❌ Repite el cálculo |
| **CTE (WITH)** | ✅ Igual (casos 1-3) | ✅ Clara y estructurada | ✅ Materializa si se reutiliza |

**Regla práctica:**
- Usa **CTE por defecto** — es más legible, más mantenible y no hay penalización de rendimiento.
- Usa **subquery** solo para expresiones simples de una línea (`WHERE id IN (SELECT ...)`).
- Si reutilizas la misma lógica más de una vez → **CTE siempre** (evita recalcular en motores que materializan).
- En BigQuery específicamente: ambos generan el mismo plan en la mayoría de casos, pero el CTE mejora mucho la experiencia del equipo.

---

> 📌 Experimento documentado en [github.com/ecartolariesteves/300-Lab](https://github.com/ecartolariesteves/300-Lab/tree/main/305-SQL-Lab/subquery-vs-cte)
>
> 💼 LinkedIn: [linkedin.com/in/edgaresteves](https://linkedin.com/in/edgaresteves)